# 03 — Стресс-тестирование
Запуск стресс-сценариев через `build_model`; covenant auto-trigger; кастомные сценарии.

**Разделы:**
1. Настройка (build_model с run_stress=True, run_covenants=True)
2. Результаты стресс-сценариев
3. Сравнение сценариев — 2025
4. Детальный анализ по годам
5. Covenant Breach — авто-триггер
6. Кастомный сценарий


## §1 Настройка

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COMPANY_ID    = 'us_steel'
SCENARIO_NAME = 'base'
DB_PATH       = ROOT / 'data_mart_v2.db'
CONFIG_PATH   = ROOT / 'companies' / COMPANY_ID / 'configs' / 'project.yaml'

import logging, warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

import pandas as pd
from engine.orchestrator import build_model
from engine.database.repository import Repository
from engine.stress.runner import StressRunner
from engine.stress.core import StressScenario, ShockSpec

# Run full pipeline with stress + covenants
result = build_model(
    company_id=COMPANY_ID, scenario_name=SCENARIO_NAME,
    db_path=DB_PATH, config_path=CONFIG_PATH,
    run_preprocessor=False, run_macro=False, run_model=True,
    run_stress=True, run_rating=False, run_covenants=True,
)
mr = result.model_result
stress_results = result.stress_results
print('Model:', 'OK' if result.success else 'FAIL')
print('Стресс-сценарии:', list(stress_results.keys()))
if result.covenants_result:
    nb_br = len(result.covenants_result.breaches())
    print(f'Ковенанты: {nb_br} нарушений')
    if 'covenant_breach' in stress_results:
        print('  [covenant_breach авто-запущен]')


## §2 Запуск стресс-сценариев

In [ ]:
# Все стресс-результаты уже доступны из build_model
print('Доступные стресс-сценарии:', list(stress_results.keys()))
print()
for sc_name, sr in stress_results.items():
    status = 'OK' if sr.success else 'FAIL'
    print(f'  {sc_name:<25} {status}')
    if not sr.success:
        print(f'    Ошибки: {sr.errors}')


## §3 Сравнение сценариев — 2025

In [ ]:
rows = []
for sc_name, sr in stress_results.items():
    if not sr.success or 2025 not in sr.comparison:
        continue
    c = sr.comparison[2025]
    rows.append({
        'Сценарий':     sc_name,
        'Rev D%':       f'{c["revenue_delta_pct"]:+.1f}%',
        'EBITDA D%':    f'{c["ebitda_delta_pct"]:+.1f}%',
        'NI D%':        f'{c["ni_delta_pct"]:+.1f}%',
        'Cash D$M':     f'{c["cash_delta"]/1e6:+.0f}' if 'cash_delta' in c else 'N/A',
        'BS diff':      f'{c.get("bs_diff", 0):.2f}',
    })

if rows:
    df = pd.DataFrame(rows).set_index('Сценарий')
    try:
        display(df)
    except:
        print(df.to_string())
else:
    print('Нет результатов для 2025')


## §4 Детальный анализ по годам

In [ ]:
# Детальный анализ по годам
SELECTED = 'steel_downturn'
sr = stress_results.get(SELECTED)
if not sr or not sr.success:
    print(f'{SELECTED}: нет результатов')
else:
    print(f'Детальный анализ: {SELECTED}')
    print()
    rows = []
    for yr in sorted(sr.comparison.keys()):
        c = sr.comparison[yr]
        rows.append({
            'Год':           yr,
            'Rev Base $B':   f'{c["revenue_base"]/1e9:.2f}',
            'Rev Stress $B': f'{c["revenue_stress"]/1e9:.2f}',
            'Rev D%':        f'{c["revenue_delta_pct"]:+.1f}%',
            'EBITDA D%':     f'{c["ebitda_delta_pct"]:+.1f}%',
            'NI Base $M':    f'{c["ni_base"]/1e6:.0f}',
            'NI Stress $M':  f'{c["ni_stress"]/1e6:.0f}',
            'NI D%':         f'{c["ni_delta_pct"]:+.1f}%',
        })
    df2 = pd.DataFrame(rows).set_index('Год')
    try:
        display(df2)
    except:
        print(df2.to_string())


## §5 Covenant Breach — авто-триггер
Когда ковенант нарушен, orchestrator авто-запускает `covenant_breach` стресс-сценарий.

In [ ]:
cov_result = result.covenants_result
if cov_result:
    breaches = cov_result.breaches()
    print(f'Нарушений ковенантов: {len(breaches)}')
    if breaches:
        print('Нарушения:')
        for b in breaches:
            print(f'  {b}')
    else:
        print('Ковенанты соблюдены в базовом сценарии')

if 'covenant_breach' in stress_results:
    sr_cb = stress_results['covenant_breach']
    print()
    print('covenant_breach стресс-сценарий (авто-запущен):')
    if sr_cb.success:
        for yr in sorted(sr_cb.comparison.keys()):
            c = sr_cb.comparison[yr]
            print(f'  {yr}: Rev={c["revenue_delta_pct"]:+.1f}% EBITDA={c["ebitda_delta_pct"]:+.1f}% NI={c["ni_delta_pct"]:+.1f}%')
    else:
        print(f'  FAIL: {sr_cb.errors}')
else:
    print()
    print('covenant_breach не авто-запущен (нарушений нет)')
    print('  Шоки сценария: steel_price_hrc -15%, steel_ppi -10%, avg_rate +2pp, DSO/DIH +10%')
    print('  Источник: companies/us_steel/configs/stress_scenarios.yaml')


## §6 Кастомный сценарий

In [ ]:
# Кастомный сценарий — запускаем через StressRunner напрямую
from engine.stress.core import StressScenario, ShockSpec

custom_scenario = StressScenario(
    name='custom_analysis',
    description='Кастомный: HRC -15%, ставка +150bp',
    macro_shocks=[
        ShockSpec(factor='steel_price_hrc', shock_type='percentage', value=-15.0),
        ShockSpec(factor='steel_ppi_iron_steel', shock_type='percentage', value=-10.0),
    ],
    driver_shocks=[
        ShockSpec(factor='avg_rate', shock_type='pp', value=1.5),
    ],
)

with Repository(db_path=DB_PATH) as repo:
    custom_runner = StressRunner(COMPANY_ID, repo, CONFIG_PATH)
    custom_sr = custom_runner.run_scenario(custom_scenario, base_scenario=SCENARIO_NAME, save=False)

print('Кастомный сценарий:', 'OK' if custom_sr.success else f'FAIL: {custom_sr.errors}')
if custom_sr.success and 2025 in custom_sr.comparison:
    c = custom_sr.comparison[2025]
    print(f'  2025: Rev={c["revenue_delta_pct"]:+.1f}% EBITDA={c["ebitda_delta_pct"]:+.1f}% NI={c["ni_delta_pct"]:+.1f}%')
